# Plot slip inversion of the synthetic GNSS displacement at Nicoya (Costa Rica) within a heterogeneous half-space, compared with a homogeneous solution

* Can deal with ground-truth slip of various pattern. In particular, checkerboard pattern. 

* The max amplitude is to simulate either the max locking (== the long-term trench-normal velocity, <100 mm/yr), or the coseismic (say a few m)

* The goal is to get a sense of how the recovery will change under the heterogeneity, i.e., can you say anything at offshore if you see something with heterogeneity?

In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
import utils_plot as utp
import meshio

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define folder to save the figures
figpath = "/home/staff/chao/SSEinv/Nicoya/figures/synth/"
os.makedirs(figpath, exist_ok=True)

# flag to indicate whether to save figures
# flag_savefig = True
flag_savefig = False

# Define the inversion results path
resultpath = "syn_slip/"

# define mesh name
# meshname = "nicoya"  # larger fault interface
# meshname = "nicoya2"   # This has a smaller fault interface
# meshname = "nicoyaCK"   # local interface model from C. Kyriakopoulos_etal2015JGRSE
# meshname = "nicoyaCK2"   # same as above but 5-km mesh size on fault
# meshname = "nicoyaCK3"   # fault zone extended to the whole subduction zone
meshname = "nicoyaCK4"   # same as CK2, but connecting the trench nowprint(meshname)

meshname2 = "nicoyaCK3_dense2_sm" 
meshname3 = "nicoyaCK3_dense7_sm"
# meshname4 = "nicoyaCK3_dense8_sm" 

print(meshname)
print(meshname2)
print(meshname3)
# print(meshname4)

# # Read the plate interface file
# plate_file = "plateinterface/nicoya_slab2_100_10_10.txt"
# df_plate = utp.parse_plate_interface_file(datadir + plate_file)
# depths = np.array(df_plate['dep'].unique())
# print("depths:", depths)

# Read the mesh file for generating the slab interface
mesh_file = "Kyriakopoulos2016JGR/Nicoya_interface.e"
mesh = meshio.read(datadir + mesh_file)
points = mesh.points  # shape (n_points, 3)
df_plate = pd.DataFrame(points, columns=["x", "y", "z"])
# define the centroid of relative coordinates that is consistent with mesh
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
df_plate['lon'], df_plate['lat'] = utp.ckm2LLd(df_plate['x'], df_plate['y'], lon0, lat0, -rot)
df_plate['z'] = df_plate['z'] /1e3  
# print(df_plate.head())

# Read the trench file
# trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_geo.txt"
# trench = pd.read_csv(trench_file, sep=r'\s+', names=['lon', 'lat'])
trench_file = "/home/staff/chao/SSEinv/Nicoya/Kyriakopoulos2016JGR/trench_xyz.txt"
trench = pd.read_csv(trench_file, sep=r'\s+', names=['x', 'y'])
trench['lon'], trench['lat'] = utp.ckm2LLd(trench['x'], trench['y'], lon0, lat0, -rot)
# print(trench.head())

# anaysis type, locking or coseismic?
type = "locking"  # "locking" or "coseismic"
# type = "coseismic"

# whethere the synthetics are polluted
pollute = True  # True or False
# pollute = False
print(pollute)

# noise std type, either 'uniform' or 'datastd'
pollute_type = 'uniform'  # uniform noise for all stations
# pollute_type = 'datastd'  # use the data standard deviation as noise std
print(pollute_type)

m2mm = 1e3  # meter to mm
km2m = 1e3   # km to m

if type == "locking":
       # read in the lon and lat of stations
       # obs_file = "data/Feng_etal_JGR_2012table1.csv" # original data from Feng et al. 2012
       obs_file = "data/Kyriakopoulos_novolcano.csv"    # same as above, but with volcano sites removed

       # note that the height is in m, Dt in years, the original displacement data is in mm/yr
       # the disp relative to the stable Caribbean plate will be used in the inversion
       # From ACOS to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
       df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                     names=['site', 'lat', 'lon', 'height', 'Dt', 'Days', \
                            'vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                            'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car'])
       df['lon'] = -1*df['lon'] # convert to east positive, as the original data is west positive
       # Convert mm to m, needed for inversion
       cols_to_convert = ['vy_ITRF05', 'vy_std_ITRF05', 'vx_ITRF05', 'vx_std_ITRF05', 'vz_ITRF05', 'vz_std_ITRF05', \
                     'vy_Car', 'vy_std_Car', 'vx_Car', 'vx_std_Car', 'vz_Car', 'vz_std_Car']
       df[cols_to_convert] = df[cols_to_convert] / m2mm  # Convert mm to m

       # displacement noise standard deviations, in m 
       if pollute:
              if pollute_type == 'uniform':
                     noise_std_h = 0.5 * (df['vx_std_Car'].mean() + df['vy_std_Car'].mean()) 
                     noise_std_v = df['vz_std_Car'].mean()
                     error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_v
                     
              elif pollute_type == 'datastd':
                     error_e, error_n, error_v = df['vx_std_Car'], df['vy_std_Car'], df['vz_std_Car']
                     
       else:
              error_e, error_n, error_v = df['vx_std_Car']*0, df['vy_std_Car']*0, df['vz_std_Car']*0

else:
       obs_file = "data/Protti_etal_2014_tableS1.csv"
       # note that the height is in m, duration and dates are in yr, and the displacements and errors are already in m
       # From BAGA to VENA are Campaign Sites; From BIJA to VERA are Continuous Sites; From AROL to WARN are Volcano Sites
       df = pd.read_csv(datadir + obs_file, sep=",", skiprows=1, \
                     names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                            'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])
       
       # displacement noise standard deviations
       if pollute:
              noise_std_h = 0.5 * (df['ux_std'].mean() + df['uy_std'].mean())
              error_e, error_n, error_v  = noise_std_h, noise_std_h, noise_std_h
       else:
              error_e, error_n, error_v = df['ux_std'], df['uy_std'], df['uz_std']

# print(df.head())  # Preview the data
print(df['lon'].min(), df['lon'].max(), df['lat'].min(), df['lat'].max())
print("Displacement noise std: ", error_e.mean(), error_n.mean(), error_v.mean())

In [ ]:
# flag to indicate if the slip inversion was bounded
BOUNDED = True
# BOUNDED = False

BOUND_TYPE = 'both'

# choose function type,  available choices are 'tanh'- Hyperbolic tangent (default), 'arctan' - Arctangent scaled by 2/π  
# 'sigmoid' - 2/(1+exp(-x)) - 1, and 'sqrt' - x/sqrt(1+x²)
FUNCTION_TYPE = 'tanh'
# FUNCTION_TYPE = 'sigmoid'

if BOUNDED:
    # Define slip bounds based on your problem
    V_para = 16.0 / m2mm
    V_norm = 78.5 / m2mm     # the trench-normal long-term loading of 78.5 mm
    if BOUND_TYPE == 'both':
        strike_bounds=(-2e-3, 2e-3)    # just a little mm. although the truth is 0
        dip_bounds=(0.0, V_norm)
        function_type=FUNCTION_TYPE
        print("Constraints to both strike and dip ")

In [ ]:
# a catalog Holocene volcanoes
volcano_file = "data/GVP_Holocene_Volcano_loc.csv" 
volcano = pd.read_csv(datadir + volcano_file, sep=",", skiprows=1, \
                      names=['id', 'lat', 'lon', 'elv'])
# Show first few rows
print(volcano.head())

region=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
# region=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
# region=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

volcano = volcano[
    (volcano['lat'] >= region[2]) & (volcano['lat'] <= region[3]) &
    (volcano['lon'] >= region[0]) & (volcano['lon'] <= region[1])
]

In [ ]:
### Choose the slip model, all models are in m_dip direction, and m_strike = 0
### All are offseted a little along dip due to the plate shape from CK
# slipmodel = 1     # along-strike 3-stripe pattern, shallow-middle-deep
slipmodel = 2     # along-strike 2-stripe pattern with gap, shallow-deep 
# slipmodel = 3     # along-strike 1-stripe pattern, complementary of model 2, middle
# slipmodel = 4     # checkerboard pattern in strike-dip direction
# slipmodel = 5     # checkerboard pattern in x and y direction  
# slipmodel = 6     # same as model 4, but amp up to 1 m rather than V_norm=78.5 mm
# slipmodel = 7     # 2D Gaussian where the contours are circular, within range of 0 and max

V_norm = 78.5 / m2mm     # the trench-normal long-term loading of 78.5 mm

# if slipmodel == 1:
#     # A checkerboard model in m_dip, alternating between 0 and amp, and m_strike = 0
#     V_norm = 78.5 / m2mm     # the trench-normal long-term loading of 78.5 mm
#     amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
#     dx = 20e3  # grid spacing in x direction, \lamda_x = 2*dx
#     dy = 20e3  # spacing in y direction, \lamda_y = 2*dy
#     x0 = 0
#     y0 = 0
#     slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_ms{amp:g}"

# elif slipmodel == 2:
#     # A checkerboard model in m_dip, alternating between 0 and amp, and m_strike = 0
#     V_norm = 78.5 / m2mm     # the trench-normal long-term loading of 78.5 mm
#     amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
#     dx = 40e3  # grid spacing in x direction, \lamda_x = 2*dx
#     dy = 40e3  # spacing in y direction, \lamda_y = 2*dy
#     # x0 = 0
#     # y0 = 0
#     x0 = -20e3
#     y0 = -20e3
#     slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_ms{amp:g}"    

if slipmodel == 1:
    # Stripe pattern in m_dip, along-strike, 3-stripe, shallow-middle-deep; m_strike = 0
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x_len = 35e3     # width of each rectangle in x direction
    dx = 25e3  # gap between rectangles
    x0 = 0e3     # x center of the pattern
    y0 = -5e3     # y center of the pattern
    # y0 = 10e3     # y center of the pattern
    # rot_deg = 45.0  # rotation angle in degrees (counter-clockwise positive)
    rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)

    slip_str_gt = (
        # f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{rot_deg:g}_ms{amp:g}"
    )

elif slipmodel == 2:
    # Stripe pattern in m_dip, along-strike, 2-stripe, shallow-deep; m_strike = 0
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x_len = 80e3     # width of each rectangle in dip direction
    y_len = 300e3   # length of each rectangle in strike direction  
    dx = 35e3  # gap between rectangles
    stripe_spacing = x_len + dx  # center-to-center distance between rectangles in dip direction
    x0 = 0     # x center of the pattern
    y0 = -45e3     # y center of the pattern
    rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)

    # y_len = 240e3   # length of each rectangle in strike direction  
    # dx = 40e3  # gap between rectangles
    # y0 = -40e3     # y center of the pattern
    # zmin = -70e3
    # zmax = 0

    slip_str_gt = (
        f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{rot_deg:g}_ms{amp:g}"
    )

elif slipmodel == 3:
    # Stripe pattern in m_dip, along-strike, 1-stripe, middle, complementary of model 2; m_strike = 0
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x_len = 40e3     # width of each rectangle in dip direction
    y_len = 300e3   # length of each rectangle in strike direction  
    dx = 100e3  # gap between rectangles
    stripe_spacing = x_len + dx  # center-to-center distance between rectangles in dip direction
    x0 = 0     # x center of the pattern
    y0 = 12.5e3     # y center of the pattern
    rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)
    
    # y_len = 240e3   # length of each rectangle in strike direction  
    # zmin = -70e3
    # zmax = 0
    
    slip_str_gt = (
        f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
        f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
        f"_rot{rot_deg:g}_ms{amp:g}"
    )

elif slipmodel == 4:
    # Checkerboard pattern in m_dip, in strike-dip direction; m_strike = 0
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    dx = 35e3  # grid spacing in x direction, \lamda_x = 2*dx
    dy = 35e3  # spacing in y direction, \lamda_y = 2*dy
    x0 = -20e3  # offset along strike
    y0 = -60e3  # offset along dip
    rot_deg = 45  # 45° counterclockwise
    zmin = -70e3
    zmax = 0
    smin = -120e3
    smax = 120e3
    slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_rot{rot_deg:g}_ms{amp:g}"    

elif slipmodel == 5:
    # Checkerboard pattern in m_dip, in x-y (N and E) direction; m_strike = 0,
    # given the strike direction is almost 45 deg, pattern is the same along dip
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    dx = 30e3  # grid spacing in x direction, \lamda_x = 2*dx
    dy = 30e3  # spacing in y direction, \lamda_y = 2*dy
    x0 = 20e3  # offset along strike
    y0 = -15e3  # offset along dip
    rot_deg = 0
    zmin = -70e3
    zmax = 0
    smin = -120e3
    smax = 120e3
    slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_rot{rot_deg:g}_ms{amp:g}"    

elif slipmodel == 6:
    # Checkerboard pattern in m_dip, in strike-dip direction; m_strike = 0
    amp = 1   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    dx = 35e3  # grid spacing in x direction, \lamda_x = 2*dx
    dy = 35e3  # spacing in y direction, \lamda_y = 2*dy
    x0 = -20e3  # offset along strike
    y0 = -60e3  # offset along dip
    rot_deg = 45  # 45° counterclockwise
    zmin = -70e3
    zmax = 0
    smin = -120e3
    smax = 120e3
    slip_str_gt = f"_check_x{x0/km2m:g}_y{y0/km2m:g}_dx{dx/km2m:g}_dy{dy/km2m:g}_rot{rot_deg:g}_ms{amp:g}"    

elif slipmodel == 7:
    # Define true model, m_strike = 0, m_dip = 2D Gaussian where the contours are circular, within range of 0 and max
    amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
    x0_lon, y0_lat = -85.5, 10     # center of the pattern, in lon-lat degrees
    x_rot, y_rot  = utp.LL2ckmd(x0_lon, y0_lat, lon0, lat0, rot)
    x0, y0 = 130e3, 350e3  # offset for x and y coordinates, in m
    x0, y0 = (x_rot - x0), (y_rot - y0)   # offset to match the mesh coordinates
    sigma = 10e3  # the Gaussian would be nearly 0 at +- 3*sigma
    slip_str_gt = f"_gauss_x{round(x0/km2m):g}_y{round(y0/km2m):g}_s{sigma/km2m:g}_rs{amp:g}"

# elif slipmodel == 7:
#     # Stripe pattern in m_dip, along-strike, 2-stripe, shallow-deep; m_strike = 0
#     amp = V_norm   # interseismic coupling case, max == complete coupling of 1 == the trench-normal long-term loading
#     x_len = 80e3     # width of each rectangle in dip direction
#     y_len = 240e3   # length of each rectangle in strike direction  
#     dx = 40e3  # gap between rectangles
#     stripe_spacing = x_len + dx  # center-to-center distance between rectangles in dip direction
#     x0 = 0e3     # x center of the pattern
#     y0 = -40e3     # y center of the pattern
#     rot_deg = 0.0  # rotation angle in degrees (counter-clockwise positive)
#     zmin = -70e3
#     zmax = 0

#     slip_str_gt = (
#         f"_stripe_x{x0/km2m:g}_y{y0/km2m:g}"
#         f"_lx{x_len/km2m:g}_dx{dx/km2m:g}"
#         f"_rot{rot_deg:g}_ms{amp:g}"
#     )

slip_str_gt1 = slip_str_gt  #with no pollution

if pollute:
    if pollute_type == 'uniform':
        slip_str_gt = slip_str_gt + "_pou"
    
    elif pollute_type == 'datastd':
        slip_str_gt = slip_str_gt + "_pod"

print("Slip model string:", slip_str_gt)    

In [ ]:
# Define the expression of the shear modulus
def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
# mu_u = mu_b
mu_upper = mu_expression(mu_u)

# shear modulus for volcanoes
# mu_v = -0.9730  # ~25 GPa
mu_v = 0
mu_volcano = mu_expression(mu_v) 

if mu_v == 0:
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}"
    # string for the contrast between slab and wedge
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
    # string for the 3d model, value multiplied by 4, relative a reference
    contrast_factor = 1.0  # amplification factor 
    het3d_str = f"_DeShon3D_ref_{round(contrast_factor)}"

else:
    print( "The shear modulus for the upper plate mu = %.1f and lower plate mu = %.1f and volcano mu = %.1f" %(mu_upper, mu_lower, mu_volcano) )
    sw_str  = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}v{round(mu_expression(mu_v))}"
    # string for the homogeneous solution
    homo_str = f"_mul{round(mu_expression(0))}u{round(mu_expression(0))}v{round(mu_expression(0))}"

print(homo_str)
print(sw_str)
print(het3d_str)

In [ ]:
# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email
# convert to relative locations in meters, and then rotate
rot = 45  # rotation angle in degrees, positive is CCW
# offset in x and y direction, the same as being done to the mesh in 'Kyriakopoulos2016JGR/convert_exodus_to_msh.ipynb'
x0, y0 = 130e3, 350e3  # offset for x and y coordinates, in m

# Load the fault geometry
def read_fault_geometry(datadir, resultpath, meshname):
    outFileName = 'fault_geometry_' + meshname + '.txt'
    loc_f = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', skiprows=lambda x: x % 3 != 1, 
                        names=['xf', 'yf', 'zf'])
    loc_f = loc_f/km2m

    # Compute the inverse projection
    loc_f['lon'], loc_f['lat'] = utp.ckm2LLd(loc_f['xf']*km2m+x0, loc_f['yf']*km2m+y0, lon0, lat0, -rot)
    print(loc_f[['lon', 'lat']].head())
    # fault.head()

    return loc_f

# Load the ground-truth slip on the fault
def read_gt_slip(datadir, resultpath, meshname, slip_str_gt, loc_f):
    
    outFileName = 'mtrue_s_fault_' + meshname + slip_str_gt + '.txt'
    print(outFileName)
    mtrue_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['s_strike', 's_dip'])
    mtrue_s['mag'] = np.sqrt(mtrue_s['s_strike']**2 + mtrue_s['s_dip']**2)
    # print(mtrue_s['s_strike'].min(), mtrue_s['s_strike'].max())
    print(mtrue_s['s_dip'].min(), mtrue_s['s_dip'].max())
    # print(mtrue_s['mag'].max())

    if type == 'locking': 
        cols_to_convert = ['s_strike', 's_dip', 'mag']
        # mtrue_s[cols_to_convert] = mtrue_s[cols_to_convert] * m2mm  # Convert m to mm
        mtrue_s[cols_to_convert] = mtrue_s[cols_to_convert] / amp    # normalized by max
        
    # print(mtrue_s['s_strike'].min(), mtrue_s['s_strike'].max())
    print(mtrue_s['s_dip'].min(), mtrue_s['s_dip'].max())
    # print(mtrue_s['mag'].max())

    mtrue_s['lon'], mtrue_s['lat'] = loc_f['lon'], loc_f['lat'] 
    
    return mtrue_s

In [ ]:
loc_f = read_fault_geometry(datadir, resultpath, meshname)
mtrue_s = read_gt_slip(datadir, resultpath, meshname, slip_str_gt, loc_f)

loc_f2 = read_fault_geometry(datadir, resultpath, meshname2)
mtrue_s2 = read_gt_slip(datadir, resultpath, meshname2, slip_str_gt, loc_f2)

loc_f3 = read_fault_geometry(datadir, resultpath, meshname3)
mtrue_s3 = read_gt_slip(datadir, resultpath, meshname3, slip_str_gt, loc_f3)

# loc_f4 = read_fault_geometry(datadir, resultpath, meshname4)
# mtrue_s4 = read_gt_slip(datadir, resultpath, meshname4, slip_str_gt)

In [ ]:
def plot_slip(mtrue_s, col2plt, region):
    
    # slipsybsz = "c0.04c"
    slipsybsz = "c0.08c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c",
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            # maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*1e3
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # fig.plot(x=[lonlt, lonrt, lonrb, lonlb, lonlt], y=[latlt, latrt, latrb, latlb, latlt], pen=True, fill="green", transparency=50)
            # grid = pygmt.xyz2grd(x=mtrue_s['lon'], y=mtrue_s['lat'], z=-mtrue_s[col2plt], region=region, spacing=(0.02, 0.02)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=mtrue_s['lon'], y=mtrue_s['lat'], z=mtrue_s[col2plt], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*1e3, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*1e3, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")            

            # fig.plot(x=df['lon'], y=df['lat'], style="s0.1c", fill="cyan", pen="0.25p,black")

            # fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style="c0.1c", fill=mtrue_s[col2plt], cmap=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude", "y+l(mm)"])

    fig.show()


plot_slip(mtrue_s, 's_dip', region)
plot_slip(mtrue_s2, 's_dip', region)
plot_slip(mtrue_s3, 's_dip', region)


In [ ]:
#coordinate rotation function
def rot_xy(x, y, rot):
    cos_rot = np.cos(np.radians(rot))
    sin_rot = np.sin(np.radians(rot))
    x_rot = x * cos_rot + y * sin_rot
    y_rot = -x * sin_rot + y * cos_rot

    return x_rot, y_rot

# Load the ground-truth moment, magnitude and potency
def read_gt_moment(datadir, resultpath, meshname, slip_str_gt, struc_str_for):
    outFileName = 'moment_true_' + meshname + slip_str_gt + struc_str_for + '.txt'
    print(outFileName)
    mom_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                      names=['moment', 'mw', 'potency'])
    return mom_true

# Load the ground-truth displacement for each forward structure model
def read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, struc_str_for):
    outFileName = 'd_obs_' + meshname + slip_str_gt + struc_str_for + '.txt'
    print(outFileName)
    u_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])

    u_true['lon'], u_true['lat'] = df['lon'], df['lat']  # add lon and lat for merging later    
    # the resulting disp aligned with mesh, reverse rotation, back to geo
    u_true['ux'], u_true['uy'] = rot_xy(u_true['ux'], u_true['uy'], -rot) 
    # vector magnitude
    u_true['mag'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()

    # print(u_true.head())

    return u_true

def read_station_grid(datadir, resultpath, meshname):
    # load the dense grid of station coordinates
    outFileName = 'dense_grid_coordinates_' + meshname + '.txt'
    print(outFileName)
    sta_grid = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                         names=['lon', 'lat', 'x', 'y', 'z'])    
    cols_to_convert = ['x', 'y', 'z']
    sta_grid[cols_to_convert] = sta_grid[cols_to_convert] * km2m  # Convert km to m, same as mesh units

    return sta_grid

# Load the ground-truth displacement for each forward structure model
def read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, struc_str_for):
    outFileName = 'd_obs_grid' + meshname + slip_str_gt + struc_str_for + '.txt'
    print(outFileName)
    u_true = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                         names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    u_true['lon'], u_true['lat'] = sta_grid['lon'], sta_grid['lat']  # add lon and lat for merging later    
    # the resulting disp aligned with mesh, reverse rotation, back to geo
    u_true['ux'], u_true['uy'] = rot_xy(u_true['ux'], u_true['uy'], -rot) 
    # vector magnitude
    u_true['mag'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2 + u_true['uz']**2)
    u_true['mag_h'] = np.sqrt(u_true['ux']**2 + u_true['uy']**2)
    u_true['mag_v'] = u_true['uz'].abs()

    # print(u_true.head())

    return u_true

# Build the difference displacement between two models
def build_diff_disp(u_1, u_2):
    u_diff = u_1.copy()
    u_diff['ux'] = u_2['ux'] - u_1['ux']
    u_diff['uy'] = u_2['uy'] - u_1['uy']
    u_diff['uz'] = u_2['uz'] - u_1['uz']
    u_diff['mag'] = u_2['mag'] - u_1['mag']
    u_diff['mag_h'] = u_2['mag_h'] - u_1['mag_h']
    u_diff['mag_v'] = u_2['mag_v'] - u_1['mag_v']
    # u_diff.head()
    return u_diff

# A different way of constructing the vectors for plotting arrows
def build_disp_vector(u_true, scaleunit, error_e=None, error_n=None, error_v=None):
    # convert from m to mm, horizontal components
    df_obs_h_data = {
        "x": u_true['lon'],
        "y": u_true['lat'],
        "east_velocity": u_true['ux']*scaleunit,
        "north_velocity": u_true['uy']*scaleunit,
    }
    
    # Add error columns only if errors are provided
    if error_e is not None and error_n is not None:
        df_obs_h_data["east_sigma"] = error_e*scaleunit
        df_obs_h_data["north_sigma"] = error_n*scaleunit
        df_obs_h_data["correlation_EN"] = 0.0
    
    df_obs_h = pd.DataFrame(data=df_obs_h_data)

    # convert from m to mm, vertical components
    df_obs_v_data = {
        "x": u_true['lon'],
        "y": u_true['lat'],
        "east_velocity": 0.0,
        "north_velocity": u_true['uz']*scaleunit,
    }
    
    # Add error columns only if errors are provided
    if error_v is not None:
        df_obs_v_data["east_sigma"] = error_v*scaleunit
        df_obs_v_data["north_sigma"] = error_v*scaleunit
        df_obs_v_data["correlation_EN"] = 0.0
    
    df_obs_v = pd.DataFrame(data=df_obs_v_data)
    
    return df_obs_h, df_obs_v

# A different way of constructing the vectors for plotting arrows on a downsampled grid
def build_disp_vector_grid(u_true, scaleunit, inc):
    """
    Downsample displacement vectors from a regular grid.
    
    Parameters:
    -----------
    df : DataFrame with 'lon', 'lat' from the original dense grid
    u_true : DataFrame with 'ux', 'uy' displacements
    inc : downsampling factor (e.g., 5 means keep every 5th point)
    
    Returns:
    --------
    vectors_h : list of vectors for PyGMT
    df_obs_h : DataFrame with downsampled data
    """
    # Determine grid dimensions from the data
    # Assuming regular grid, find unique values
    unique_lons = u_true['lon'].unique()
    unique_lats = u_true['lat'].unique()
    n_lon = len(unique_lons)
    n_lat = len(unique_lats)
    
    print(f"Detected grid: {n_lat} x {n_lon}")
    
    # Reshape to 2D arrays
    ux_2d = u_true['ux'].to_numpy().reshape(n_lat, n_lon)
    uy_2d = u_true['uy'].to_numpy().reshape(n_lat, n_lon)
    lon_2d = u_true['lon'].to_numpy().reshape(n_lat, n_lon)
    lat_2d = u_true['lat'].to_numpy().reshape(n_lat, n_lon)
    
    # Downsample in 2D (keeps regular spacing)
    ux_sub = ux_2d[::inc, ::inc]
    uy_sub = uy_2d[::inc, ::inc]
    lon_sub = lon_2d[::inc, ::inc]
    lat_sub = lat_2d[::inc, ::inc]
    
    # Flatten back to 1D
    ux = ux_sub.flatten()
    uy = uy_sub.flatten()
    
    # Calculate vector angle and length
    angle = np.degrees(np.arctan2(uy, ux))
    length = np.hypot(ux, uy)
    
    print(f"Downsampled to: {ux_sub.shape[0]} x {ux_sub.shape[1]} = {len(ux)} points")
    
    # Create DataFrame for plotting
    #format for 'pygmt.Figure.velo'
    df_obs_h_velo = pd.DataFrame(
        data={
            "x": lon_sub.flatten(),
            "y": lat_sub.flatten(),
            "east_velocity": ux * scaleunit,
            "north_velocity": uy * scaleunit,
        }
    )
    
    #format for 'pygmt.Figure.plot'
    df_obs_h_plot = pd.DataFrame({
        'x': lon_sub.flatten(),
        'y': lat_sub.flatten(),
        'angle': angle,
        'length': length * scaleunit,  # convert from m to a customized unit by scaling
    })


    return df_obs_h_velo, df_obs_h_plot 

In [ ]:
def station_grid(regiongrid, grid_spacing_deg, depth_levels):
    lon_min, lon_max = regiongrid[0], regiongrid[1]  # degrees longitude
    lat_min, lat_max = regiongrid[2], regiongrid[3]    # degrees latitude

    # Create regular lat/lon meshgrid
    lon_grid = np.arange(lon_min, lon_max + grid_spacing_deg, grid_spacing_deg)
    lat_grid = np.arange(lat_min, lat_max + grid_spacing_deg, grid_spacing_deg)
    LON_GRID, LAT_GRID = np.meshgrid(lon_grid, lat_grid)

    # Flatten for processing
    lon_2d = LON_GRID.flatten()
    lat_2d = LAT_GRID.flatten()

    # Convert 2D grid to relative coordinates
    x_rot_2d, y_rot_2d = utp.LL2ckmd(lon_2d, lat_2d, lon0, lat0, rot)
    x_2d = (x_rot_2d - x0) / 1e3  # convert to km
    y_2d = (y_rot_2d - y0) / 1e3

    # Replicate the 2D grid for each depth level
    n_2d = len(x_2d)
    n_depths = len(depth_levels)
    n_total = n_2d * n_depths

    lon_dense = np.tile(lon_2d, n_depths)
    lat_dense = np.tile(lat_2d, n_depths)
    x_dense = np.tile(x_2d, n_depths)
    y_dense = np.tile(y_2d, n_depths)
    z_dense = np.repeat(depth_levels, n_2d)  # repeat each depth n_2d times

    # Create dense grid dataframe
    sta_grid = pd.DataFrame({
        'lon': lon_dense,
        'lat': lat_dense,
        'x': x_dense,
        'y': y_dense,
        'z': z_dense
    })

    return sta_grid

# Define regular lat/lon grid covering study area
# Based on your image, approximate coverage around Costa Rica/Nicaragua region
# regiongrid=[-87, -84, 8.5, 11.5]    # suitable region for chopping the plate interface grid file 
# regiongrid=[-86.75, -84.4, 8.75, 11.25]    # suitable region for chopping the plate interface grid file 
# regiongrid=[-88, -83, 7.5, 12.5] 
regiongrid=[-87.5, -83.5, 8, 12] 
# regiongrid=[-88, -83, 6, 14]    # suitable region for chopping the plate interface grid file 

# Grid resolution - adjust as needed for your visualization requirements
grid_spacing_deg = 0.01  # ~2 km spacing at this latitude
# grid_spacing_deg = 0.1  # ~20 km spacing at this latitude

# Define depth levels (0 to -80 km with 10 km increment)
# depth_levels = np.arange(0, -80 - 10, -10)  # [0, -10, -20, ..., -80]
depth_levels = [0]
print(f"Depth levels: {depth_levels} km")

sta_grid = station_grid(regiongrid, grid_spacing_deg, depth_levels)    
# print(sta_grid.head())

In [ ]:
depth_slice = 0.0  # at surface

# # read the locations of the dense grid 
# sta_grid = read_station_grid(datadir, resultpath, meshname2)
# # print(sta_grid.head())

# build observation disp. vectors at each grid point, downsampled
inc = 20    #increment, or downsample rate

u_true_3d_grid = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname, slip_str_gt, het3d_str)
u_true_3d_grid = u_true_3d_grid[u_true_3d_grid['z'] == depth_slice]
print(u_true_3d_grid['uz'].min()*m2mm, u_true_3d_grid['uz'].max()*m2mm )
print(u_true_3d_grid['mag_h'].min()*m2mm, u_true_3d_grid['mag_h'].max()*m2mm )
df_obs_h_3d_grid, _ = build_disp_vector_grid(u_true_3d_grid, m2mm, inc)

u_true_3d2_grid = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname2, slip_str_gt, het3d_str)
u_true_3d2_grid = u_true_3d2_grid[u_true_3d2_grid['z'] == depth_slice]
df_obs_h_3d2_grid, _ = build_disp_vector_grid(u_true_3d2_grid, m2mm, inc)

u_true_3d3_grid = read_gt_disp_grid(sta_grid, datadir, resultpath, meshname3, slip_str_gt, het3d_str)
u_true_3d3_grid = u_true_3d3_grid[u_true_3d3_grid['z'] == depth_slice]
df_obs_h_3d3_grid, _ = build_disp_vector_grid(u_true_3d3_grid, m2mm, inc)

# difference in displacements
u_true_3d2diff_grid = build_diff_disp(u_true_3d_grid, u_true_3d2_grid)
df_obs_h_3d2diff_grid, _ = build_disp_vector_grid(u_true_3d2diff_grid, m2mm, inc)

u_true_3d3diff_grid = build_diff_disp(u_true_3d_grid, u_true_3d3_grid)
df_obs_h_3d3diff_grid, _ = build_disp_vector_grid(u_true_3d3diff_grid, m2mm, inc)

u_true_3d23diff_grid = build_diff_disp(u_true_3d2_grid, u_true_3d3_grid)
df_obs_h_3d23diff_grid, _ = build_disp_vector_grid(u_true_3d23diff_grid, m2mm, inc)


In [ ]:
def plot_true_slip_disp_grid(mtrue_s, col2plt, scale_vec, df_obs_h_grid, u_true_grid, region, struc_str_for, flag_savefig=False, df_obs_h_noerr=None):

    slipsybsz = "c0.08c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", 
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            # fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style="c0.1c", fill=mtrue_s[col2plt], cmap=True)
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic disp."])
            # use the vertical displacement as background color
            griduv = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['uz']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
            uz_max = u_true_grid['uz'].abs().max()*m2mm     # in mm
            print(uz_max)
            # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
            # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
            maxdisp = uz_max
            pygmt.makecpt(cmap='roma', series=[-maxdisp, maxdisp, maxdisp/10], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
            fig.grdimage(region=region, projection="M?", grid=griduv, cmap=True, nan_transparent=True, interpolation="l")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lVert. disp.", "y+l(mm)"]) #

            fig.coast(borders=None, shorelines="0.1p,black", area_thresh=20)

            # plot the contour lines of horizontal displacement vector magnitude
            # griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
            # fig.grdcontour(grid=griduh, annotation="+f4p, white", pen="0.25p, white") #levels=2, 2

            # scale_vec = 0.05    # 1 mm would be scaled by 'scale_vec'
            # fig.plot(region=region, projection="M?", x=df_obs_h_grid['x'], y=df_obs_h_grid['y'],
            #          direction=[df_obs_h_grid['angle'], df_obs_h_grid['length']*scale_vec],
            #          style='v0.1c+e+jb+h0+a45', pen='0.5p,black', fill='white')     # data=df_obs_h_grid,
            # fig.velo(data=lgd, pen="0.5p,black", spec="e"+str(scale_vec), vector="0.1c+a45+p0.5p,black+ea+gwhite")
            
            #plot the horizontal displacement vectors over the grid, error-free
            fig.velo(data=df_obs_h_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            #plot legends
            fig.plot(x=region[0]+0.35, y=region[-2]+0.2, style="r1/0.5", fill="white", pen=None, transparency=30, )
            lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
            fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
            fig.text(text=str(int(scale_vec))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")

        # direct comparison with error-free displacement vectors at real station locations 
        if df_obs_h_noerr is not None:
            with fig.set_panel(panel=[0, 2]):
                fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tCompare at real stations"])
                #plot the horizontal displacement vectors over the grid, error-free
                fig.velo(data=df_obs_h_grid, pen="0.5p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                        vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
                #plot the horizontal displacement vectors over real station, error-free
                fig.velo(data=df_obs_h_noerr, pen="0.5p,magenta", spec="e"+str(0.5/scale_vec)+"/0.39", 
                                    vector="0.1c+a45+p0.1p,magenta+ea+gmagenta+h0") 
                # plot legends
                fig.plot(x=region[0]+0.35, y=region[-2]+0.2, style="r1/0.5", fill="white", pen=None, transparency=30, )
                lgd = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.3, "east_velocity": [1], "north_velocity": [0]})
                fig.velo(data=lgd, pen="0.5p,black", spec="e0.5/0.39", vector="0.1c+a45+p0.1p,black+ea+gwhite+h0")
                fig.text(text=str(int(scale_vec))+" mm", x=region[0]+0.1, y=region[-2]+0.15, font="6p,Helvetica,black", justify="ML")                

    fig.show()

    if flag_savefig: 
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_trueslip.pdf")

uh_max = u_true_3d_grid['mag_h'].max()*m2mm
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = 5*n    # vector length are plotted relative to scale_vec*length in original unit, here mm 
print(uh_max, scale_vec)

# #plot the true slip model, and resulting disp vectors over a regular grid under the respective structure models, compare them at real stations without error 
# plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_hom_grid, u_true_hom_grid, region, homo_str, flag_savefig=False, df_obs_h_noerr=df_obs_h_hom1)

#plot the true slip model, and resulting disp vectors over a regular grid under the respective structure models
plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_3d_grid, u_true_3d_grid, region, het3d_str, flag_savefig=False)
plot_true_slip_disp_grid(mtrue_s2, 's_dip', scale_vec, df_obs_h_3d2_grid, u_true_3d2_grid, region, het3d_str, flag_savefig=False)
plot_true_slip_disp_grid(mtrue_s3, 's_dip', scale_vec, df_obs_h_3d3_grid, u_true_3d3_grid, region, het3d_str, flag_savefig=False)


In [ ]:
#plot the differential disp vectors between models over a regular grid
scale_vec = 5
plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_3d2diff_grid, u_true_3d2diff_grid, region, het3d_str, flag_savefig=False)
plot_true_slip_disp_grid(mtrue_s, 's_dip', scale_vec, df_obs_h_3d23diff_grid, u_true_3d23diff_grid, region, het3d_str, flag_savefig=False)


In [ ]:
# 3D DeShon model, value multiplied by 4, relative to a reference
u_true_3d = read_gt_disp(df, datadir, resultpath, meshname, slip_str_gt, het3d_str)
df_obs_h_3d, df_obs_v_3d = build_disp_vector(u_true_3d, m2mm, error_e, error_n, error_v)
mom_true_3d = read_gt_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str)
print(u_true_3d[['lon', 'lat', 'ux', 'uy', 'uz']].head())

u_true_3d2 = read_gt_disp(df, datadir, resultpath, meshname2, slip_str_gt, het3d_str)
df_obs_h_3d2, df_obs_v_3d2 = build_disp_vector(u_true_3d2, m2mm, error_e, error_n, error_v)
mom_true_3d2 = read_gt_moment(datadir, resultpath, meshname2, slip_str_gt, het3d_str)
print(u_true_3d2[['lon', 'lat', 'ux', 'uy', 'uz']].head())

u_true_3d3 = read_gt_disp(df, datadir, resultpath, meshname3, slip_str_gt, het3d_str)
df_obs_h_3d3, df_obs_v_3d3 = build_disp_vector(u_true_3d3, m2mm, error_e, error_n, error_v)
mom_true_3d3 = read_gt_moment(datadir, resultpath, meshname3, slip_str_gt, het3d_str)
print(u_true_3d3[['lon', 'lat', 'ux', 'uy', 'uz']].head())

# u_true_3d4 = read_gt_disp(df, datadir, resultpath, meshname4, slip_str_gt, het3d_str)
# df_obs_h_3d4, df_obs_v_3d4 = build_disp_vector(u_true_3d, m2mm, error_e, error_n, error_v)
# mom_true_3d4 = read_gt_moment(datadir, resultpath, meshname4, slip_str_gt, het3d_str)

u_true_3ddiff2 = build_diff_disp(u_true_3d, u_true_3d2)
print(u_true_3ddiff2[['lon', 'lat', 'ux', 'uy', 'uz']].head())
df_obs_h_3ddiff2, df_obs_v_3ddiff2 = build_disp_vector(u_true_3ddiff2, m2mm, error_e, error_n, error_v)

u_true_3ddiff3 = build_diff_disp(u_true_3d, u_true_3d3)
df_obs_h_3ddiff3, df_obs_v_3ddiff3 = build_disp_vector(u_true_3ddiff3, m2mm, error_e, error_n, error_v)

u_true_3ddiff23 = build_diff_disp(u_true_3d2, u_true_3d3)
print(u_true_3ddiff23[['lon', 'lat', 'ux', 'uy', 'uz']].head())
df_obs_h_3ddiff23, df_obs_v_3ddiff23 = build_disp_vector(u_true_3ddiff23, m2mm, error_e, error_n, error_v)

In [ ]:
def plot_true_slip_disp(mtrue_s, col2plt, scale_vec, df_obs_h, df_obs_v, region, struc_str_for, flag_savefig=False, u_true_grid=None):

    slipsybsz = "c0.08c"
    # slipsybsz = "c0.05c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", 
                    MAP_SCALE_HEIGHT="3p", FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            maxslip = mtrue_s[col2plt].max()
            maxslip = 1*amp*m2mm
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*amp*m2mm, cmap=True)   # no smoothing or interpolation
            # fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=mtrue_s[col2plt]*m2mm, cmap=True)   # no smoothing or interpolation
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")
            # fig.plot(x=-84.869, y=10.8536, style="s0.15c", fill="red", pen="0.25p,black")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lSlip mag.", "y+l(mm)"])

        # Synthetic horizontal displacement
        with fig.set_panel(panel=[0, 1]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic horiz. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)   #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            if u_true_grid is not None:
                griduh = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
                # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
                # data_bmedian = pygmt.blockmedian(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_h']*m2mm, region=region, spacing=(0.01, 0.01))
                # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
                maxdisp = u_true_grid['mag_h'].max()*m2mm     # in mm
                # maxdisp = max(u_true_grid['mag_h'].max()*m2mm, u_true_grid['mag_v'].max()*m2mm)     # in mm
                # maxdisp = u_true_grid['mag'].max()*m2mm     # in mm
                pygmt.makecpt(cmap=colormap, series=[0, maxdisp, maxdisp/20], background=True, reverse=True)
                # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
                fig.grdimage(grid=griduh, cmap=True, nan_transparent=True, interpolation="l")
                with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", "x+lHoriz. disp. mag.", "y+l(mm)"]) #
                # fig.grdcontour(grid=griduh, levels=5, annotation="5+f4p, black", pen="0.25p, black") #

            #Synthetic displacement
            fig.velo(data=df_obs_h, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39", 
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.15, "y": region[-2]+0.5, "east_velocity": [1], "north_velocity": [0],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs H", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML")

        # Synthetic vertical displacement
        with fig.set_panel(panel=[0, 2]):
            fig.coast(region=region, projection="M?", frame=["a1f0.2", "+tSynthetic vert. disp."], borders=None, shorelines="0.25p,black",
                      area_thresh=20, land="lightgray", water=None, resolution="h", map_scale=mapscale_str)  #land="251/250/240", water="241/246/247"
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            if u_true_grid is not None:
                griduz = pygmt.xyz2grd(x=u_true_grid['lon'], y=u_true_grid['lat'], z=u_true_grid['mag_v']*m2mm, region=region, spacing=(0.01, 0.01)) # no smoothing
                # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
                # data_bmedian = pygmt.blockmedian(x=sta_grid['lon'], y=sta_grid['lat'], z=u_true_grid['mag_v']*m2mm, region=region, spacing=(0.01, 0.01))
                # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.01, 0.01))
                maxdisp = u_true_grid['mag_v'].max()*m2mm     # in mm
                pygmt.makecpt(cmap=colormap, series=[0, maxdisp, maxdisp/20], background=True, reverse=True)
                # pygmt.makecpt(cmap=colormap, series=[0, maxdisp], background=True, reverse=True)
                fig.grdimage(grid=griduz, cmap=True, nan_transparent=True, interpolation="l")
                with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                    fig.colorbar(frame=["af", "x+lVert. disp. mag.", "y+l(mm)"]) # 
                # fig.grdcontour(grid=griduz, levels=5, annotation="5+f4p, black", pen="0.25p, black") #
                    
            #Synthetic displacement
            fig.velo(data=df_obs_v, pen="0.5p,black", uncertaintyfill=None, line="0.25p,black", spec="e"+str(0.5/scale_vec)+"/0.39",
                     vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot legends
            fig.plot(x=region[0]+0.6, y=region[-2]+0.5, style="r2.6/2", fill="white", pen="0.5p,black", transparency=30, )
            fig.plot(x=region[0]+0.3, y=region[-2]+0.8, style="s0.15c", fill="cyan", pen="0.25p,black")
            lgdobs = pd.DataFrame(data={"x": region[0]+0.4, "y": region[-2]+0.3, "east_velocity": [0], "north_velocity": [1],
                                  "east_sigma": 1/5, "north_sigma": 1/5, "correlation_EN": 0.0, })
            fig.velo(data=lgdobs, pen="0.5p,black", line="0.25p,black", spec="e0.5/0.39", vector="0.1c+a45+p1p,black+ea+gblack+h0")
            #plot texts
            fig.text(text="Stations", x=region[0]+0.6, y=region[-2]+0.8, font="8p,Helvetica,black", justify="ML")
            fig.text(text="Obs V", x=region[0]+0.6, y=region[-2]+0.5, font="8p,Helvetica,black", justify="ML")
            fig.text(text=str(int(scale_vec))+"±"+str(int(scale_vec/5))+" mm", x=region[0]+0.1, y=region[-2]+0.2, font="8p,Helvetica,black", justify="ML") 


    fig.show()

    if flag_savefig: 
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_trueslip.pdf")


uh_max = np.hypot(df_obs_h_3d['east_velocity'].to_numpy(), df_obs_h_3d['north_velocity'].to_numpy()).max()
n = np.ceil(uh_max / 10)  # get the n where scale_vec=5*n would make the ratio between disp and scale_vec is less than 2
scale_vec = np.round(5*n)    # vector length are plotted relative to scale_vec*length in original unit, here mm 
print(uh_max, scale_vec)

# plot the true slip model, and resulting disp vectors under the respective structure models
plot_true_slip_disp(mtrue_s, 's_dip', scale_vec, df_obs_h_3d, df_obs_v_3d, region, het3d_str, flag_savefig=False)
plot_true_slip_disp(mtrue_s2, 's_dip', scale_vec, df_obs_h_3d2, df_obs_v_3d2, region, het3d_str, flag_savefig=False)
plot_true_slip_disp(mtrue_s3, 's_dip', scale_vec, df_obs_h_3d3, df_obs_v_3d3, region, het3d_str, flag_savefig=False)
# plot_true_slip_disp(mtrue_s, 's_dip', scale_vec, df_obs_h_3d4, df_obs_v_3d4, region, het3d_str, flag_savefig=False)

scale_vec = 5
plot_true_slip_disp(mtrue_s, 's_dip', scale_vec, df_obs_h_3ddiff2, df_obs_v_3ddiff2, region, het3d_str, flag_savefig=False)
plot_true_slip_disp(mtrue_s, 's_dip', scale_vec, df_obs_h_3ddiff23, df_obs_v_3ddiff23, region, het3d_str, flag_savefig=False)


In [ ]:
# Decide the weights of the horizontal, vertical components
if pollute:
    if pollute_type == 'uniform':   
        # Decide the weights of the horizontal, vertical components
        f_h, f_v = 1, 1 
        # f_h, f_v = 1, 1/2 
        # Take the inverse for saving the name of the weights
        w_h, w_v = int(1/noise_std_h), int(1/noise_std_v)
        
    elif pollute_type == 'datastd':
        # Decide the weights of the horizontal, vertical components
        f_h, f_v = 1, 1
        # f_h, f_v = 1, 1/2
        # Take the inverse for saving the name of the weights
        w_h, w_v = int(1/f_h), int(1/f_v)
else:
    # Decide the weights of the horizontal, vertical components
    f_h, f_v = 1, 1
    # Print the weights of the data
    print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )

    # Take the inverse for saving the name of the weights
    w_h, w_v = int(1/f_h), int(1/f_v)


In [ ]:
# Define regularization weights
# In a Bayesian inference setting, the ratio \rho = \sqrt(\gamma/\delta) plays the role of the correlation length in the prior term.
# For our case, the station separation is around 20 km, and the mesh size on the fault is 4-20 km 
rho_s = 1e9   # allows variations of slip of the order of ~30 km 

# rho_s = 1e8   # allows variations of slip of the order of ~10 km, close to the maximum resolution

if pollute:
    if pollute_type == 'uniform':   
        # gammas_s = [1e2, 2e2, 4e2, 8e2, 1e3]
        if slipmodel == 4 or slipmodel == 5 or slipmodel == 6:  # checkerboard models
            gammas_s = [4e2, 6e2, 8e2, 1e3]
            gammas_s = [1e1, 2e1, 4e1, 5e1, 1e2, 1e3]
            gamma_val_H1 = gammas_s[3]
            gamma_val_H1 = 2.5e1
        elif slipmodel == 1 or slipmodel == 2 or slipmodel == 3:   # stripe models
            gammas_s = [8e2, 1e3, 4e3, 6e3, 8e3]
            # gamma_val_H1 = gammas_s[1]
            gammas_s = [1e1, 5e1, 1e2, 2e2, 4e2]
            gamma_val_H1 = gammas_s[2]
            # gamma_val_H1 = 6e1

        # gamma_val_H1 = 8e2  # 8e2 looks best  
        # gamma_val_H1 = 1e3
        # gamma_val_H1 = 4e3

    elif pollute_type == 'datastd':
        gammas_s = [1e2, 2e2, 4e2, 8e2, 1e3, 2e3, 4e3, 8e3]
        gamma_val_H1 = gammas_s[5]
        # gamma_val_H1 = 2e3  # 2e3 looks best  
        
else:
    gammas_s = [1e-2, 1e-1, 1e0, 1e1, 4e1, 1e2, 2e2, 4e2, 8e2, 1e3]
    # gamma_val_H1 = gammas_s[1]
    gamma_val_H1 = 1e-1  # 1e-1 looks best

print("slip model number:", slipmodel)
print(pollute_type)
# gammas_s = [1e1, 2.5e1, 5e1, 1e2, 5e2, 1e3]
# gamma_val_H1 = gammas_s[2]
delta_val_L2 = gamma_val_H1 / rho_s  
# delta_val_L2 = 2.5e-7

# gamma_val_H1 = 1e3  
# delta_val_L2 = 1e-5
# gamma_val_H1 = 2.5e3  
# delta_val_L2 = 2.5e-5
# gamma_val_H1 = 5e3  

# gamma_val_H1 = 1e3  
# delta_val_L2 = 1e-6
# gamma_val_H1 = 5e2  
# delta_val_L2 = 1e-6
# gamma_val_H1 = 2.5e2  
# delta_val_L2 = 5e-6
# gamma_val_H1 = 7.5e3  
# delta_val_L2 = 7.5e-6
# gamma_val_H1 = 1e4  
# delta_val_L2 = 2.5e-6
# gamma_val_H1 = 7.5e2  
# delta_val_L2 = 5e-5

# gamma_val_H1 = 1e3  
# delta_val_L2 = 1e-4
# gamma_val_H1 = 2.5e3  
# delta_val_L2 = 2.5e-5
# delta_val_L2 = 5e-5
# delta_val_L2 = 7.5e-5
# delta_val_L2 = 1e-4

print(f"{gamma_val_H1:.0e}")
print(f"{delta_val_L2:.0e}")

# file identifier
if type == "locking":
    if BOUNDED:
        inv_str = f"_synlockbd_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
        if FUNCTION_TYPE == 'sigmoid':
            inv_str = f"_synlockbd{FUNCTION_TYPE[:4]}_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
    else:
        inv_str = f"_synlock_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"

elif type == "coseismic":    
    inv_str = f"_syncoseis_w{w_h}{w_v}_gs{gamma_val_H1:.0e}_ds{delta_val_L2:.0e}"
    
print(inv_str)


In [ ]:
# Load the predicted moment, magnitude and potency
def read_pred_moment(datadir, resultpath, meshname, slip_str_gt, \
                     struc_str_for, inv_str, struc_str_inv):
    outFileName = 'moment_' + meshname +  slip_str_gt + struc_str_for + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    mom = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                      names=['moment', 'mw', 'potency'])
    return mom

# Load the predicted surface displacement, all in meters
def read_pred_disp(u_true, datadir, resultpath, meshname, slip_str_gt, \
                   struc_str_for, inv_str, struc_str_inv):
    outFileName = 'd_cal_' + meshname + slip_str_gt + struc_str_for + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    u_pred = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                        names=['x', 'y', 'z', 'ux', 'uy', 'uz'])
    
    # the resulting disp aligned with mesh, reverse rotation, back to geo
    u_pred['ux'], u_pred['uy'] = rot_xy(u_pred['ux'], u_pred['uy'], -rot) 

    u_res = u_pred.copy()
    u_res['ux'] = u_true['ux'] - u_pred['ux']
    u_res['uy'] = u_true['uy'] - u_pred['uy']
    u_res['uz'] = u_true['uz'] - u_pred['uz']
    u_res['mag'] = np.sqrt(u_res['ux']**2 + u_res['uy']**2 + u_res['uz']**2)
    # u_res.head()

    return u_pred, u_res

# Load the inferred slip on the fault, all in meters
def read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, loc_f, \
                       struc_str_for, inv_str, struc_str_inv, type):
    outFileName = 'm_s_fault_' + meshname + slip_str_gt + struc_str_for + inv_str + struc_str_inv + '.txt'
    print(outFileName)
    m_s = pd.read_csv(datadir + resultpath + outFileName, sep=r'\s+', 
                    names=['s_strike', 's_dip'])
    m_s['mag'] = np.sqrt(m_s['s_strike']**2 + m_s['s_dip']**2)
    # print(m_s['s_strike'].min(), m_s['s_strike'].max())
    # print(m_s['s_dip'].min(), m_s['s_dip'].max())
    # print(m_s['mag'].max())

    if type == 'locking': 
        cols_to_convert = ['s_strike', 's_dip', 'mag']
        # m_s[cols_to_convert] = m_s[cols_to_convert] * 1e3  # Convert m to mm
        m_s[cols_to_convert] = m_s[cols_to_convert] / amp    # normalized by max
    # print(m_s['s_strike'].min(), m_s['s_strike'].max())
    print(m_s['s_dip'].min(), m_s['s_dip'].max())
    # print(m_s['mag'].max())

    m_s['lon'], m_s['lat'] = loc_f['lon'], loc_f['lat'] 
  
    return m_s

In [ ]:
### Load the predicted surface displacements and residuals for diff forward and inversion models

## 3d forward, hom inversion
m_s_3dhom = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, loc_f, \
                                het3d_str, inv_str, homo_str, type)
mom_3dhom = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, homo_str)

## 3d forward, 3d inversion
m_s_3d3d = read_inferred_slip(datadir, resultpath, meshname, slip_str_gt, loc_f, \
                                het3d_str, inv_str, het3d_str, type)
mom_3d3d = read_pred_moment(datadir, resultpath, meshname, slip_str_gt, het3d_str, inv_str, het3d_str)

# 
m_s_3dhom2 = read_inferred_slip(datadir, resultpath, meshname2, slip_str_gt, loc_f2, \
                                het3d_str, inv_str, homo_str, type)
mom_3dhom2 = read_pred_moment(datadir, resultpath, meshname2, slip_str_gt, het3d_str, inv_str, homo_str)
m_s_3d3d2 = read_inferred_slip(datadir, resultpath, meshname2, slip_str_gt, loc_f2, \
                                het3d_str, inv_str, het3d_str, type)
mom_3d3d2 = read_pred_moment(datadir, resultpath, meshname2, slip_str_gt, het3d_str, inv_str, het3d_str)

# 
m_s_3dhom3 = read_inferred_slip(datadir, resultpath, meshname3, slip_str_gt, loc_f3, \
                                het3d_str, inv_str, homo_str, type)
mom_3dhom3 = read_pred_moment(datadir, resultpath, meshname3, slip_str_gt, het3d_str, inv_str, homo_str)
m_s_3d3d3 = read_inferred_slip(datadir, resultpath, meshname3, slip_str_gt, loc_f3, \
                                het3d_str, inv_str, het3d_str, type)
mom_3d3d3 = read_pred_moment(datadir, resultpath, meshname3, slip_str_gt, het3d_str, inv_str, het3d_str)

# 
# u_pred_3d3d4, u_res_3d3d4 = read_pred_disp(u_true_3d4, datadir, resultpath, meshname4, slip_str_gt, \
#                                              het3d_str, inv_str, het3d_str)
# m_s_3d3d4 = read_inferred_slip(datadir, resultpath, meshname4, slip_str_gt, \
#                                 het3d_str, inv_str, het3d_str, type)
# mom_3d3d4 = read_pred_moment(datadir, resultpath, meshname4, slip_str_gt, het3d_str, inv_str, het3d_str)



In [ ]:
### For testing, the GT potency for diff structure models should be the same
print(mom_true_3d['potency'].to_numpy())
print(mom_true_3d2['potency'].to_numpy())
print(mom_true_3d3['potency'].to_numpy())
# print(mom_true_3d4['potency'].to_numpy())


In [ ]:
### For testing, the GT potency for diff structure models should be the same
print(mom_3d3d['potency'].to_numpy())
print(mom_3d3d2['potency'].to_numpy())
print(mom_3d3d3['potency'].to_numpy())
# print(mom_3d3d4['potency'].to_numpy())

In [ ]:
assess_col = 's_dip'   # assessment component

In [ ]:
# # if slipmodel == 1 or slipmodel == 2 or slipmodel == 3:
# # Global assessment on the recovery
# gmetrics_homhom = utp.GlobalSlipMetrics(mtrue_s, m_s_homhom, mtrue_s)
# gmetrics_swsw = utp.GlobalSlipMetrics(mtrue_s, m_s_swsw, mtrue_s)
# gmetrics_3d3d = utp.GlobalSlipMetrics(mtrue_s, m_s_3d3d, mtrue_s)

# print('Global recovery assessment: ')
# recovery_thresh = 0.8   # threshold to take as recovered, in fraction

# print('-- From Homogeneous inversion -- ')
# print(gmetrics_homhom.summary(col=assess_col, threshold=recovery_thresh))

# print('-- From Slab/wedge contrast inversion -- ')
# print(gmetrics_swsw.summary(col=assess_col, threshold=recovery_thresh))

# print('-- From 3D model inversion -- ')
# print(gmetrics_3d3d.summary(col=assess_col, threshold=recovery_thresh))

In [ ]:
# if slipmodel == 1 or slipmodel == 2 or slipmodel == 3:
#     # Individual assessment on the recovery
#     metrics_homhom = utp.SlipRecoveryMetrics(mtrue_s, m_s_homhom, mtrue_s)
#     metrics_swsw = utp.SlipRecoveryMetrics(mtrue_s, m_s_swsw, mtrue_s)
#     metrics_3d3d = utp.SlipRecoveryMetrics(mtrue_s, m_s_3d3d, mtrue_s)

#     if slipmodel == 2:
#         ## offshore stripe
#         mask_shallow = (mtrue_s[assess_col] == 1.0) & (mtrue_s['zf'] > -25)
#         print('Offshore stripe recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_shallow, col=assess_col, threshold=recovery_thresh))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_shallow, col=assess_col, threshold=recovery_thresh))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_shallow, col=assess_col, threshold=recovery_thresh))

#         ## onshore stripe
#         mask_deep = (mtrue_s[assess_col] == 1.0) & (mtrue_s['zf'] < -25)
#         print('Onshore stripe recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_deep, col=assess_col, threshold=recovery_thresh))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_deep, col=assess_col, threshold=recovery_thresh))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_deep, col=assess_col, threshold=recovery_thresh))

#         ## gap in between 
#         mask_gap = (mtrue_s[assess_col] < 1e-5) & (mtrue_s['zf'] > -50) & (mtrue_s['zf'] < -10)
#         print('Gap recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_gap, col=assess_col, type='gap'))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_gap, col=assess_col, type='gap'))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_gap, col=assess_col, type='gap'))
    
#     elif slipmodel == 3:
#         ## offshore gap
#         mask_gap1 = (mtrue_s[assess_col] < 1e-5) & (mtrue_s['zf'] > -25)
#         print('Offshore gap recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_gap1, col=assess_col, type='gap'))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_gap1, col=assess_col, type='gap'))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_gap1, col=assess_col, type='gap'))

#         ## onshore gap
#         mask_gap2 = (mtrue_s[assess_col] < 1e-5) & (mtrue_s['zf'] < -25)
#         print('Onshore gap recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_gap2, col=assess_col, type='gap'))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_gap2, col=assess_col, type='gap'))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_gap2, col=assess_col, type='gap'))

#         ## intermediate stripe 
#         mask_inter = (mtrue_s[assess_col] == 1.0)
#         print('Intermediate stripe recovery assessment: ')

#         print('-- From Homogeneous inversion -- ')
#         print(metrics_homhom.summary(mask_inter, col=assess_col, threshold=recovery_thresh))

#         print('-- From Slab/wedge contrast inversion -- ')
#         print(metrics_swsw.summary(mask_inter, col=assess_col, threshold=recovery_thresh))

#         print('-- From 3D model inversion -- ')
#         print(metrics_3d3d.summary(mask_inter, col=assess_col, threshold=recovery_thresh))

In [ ]:
def plot_resolution_comparison(mtrue_s, m_s_3d, col2plt, region, absflag=False, flag_savefig=False):

    slipsybsz = "c0.09c"
    # slipsybsz = "c0.03c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the prescribed true slip model, and the synthetic displacement from the forward modeling 
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=2, figsize=("10c", "6c"), autolabel=False, frame=["a1f0.2", "WSne"], 
                    margins=["0.1c", "0.2c"],sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")
        
        # True slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tTrue slip"])
            # maxslip = mtrue_s[col2plt].max()
            maxslip = 1
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # fig.plot(x=[lonlt, lonrt, lonrb, lonlb, lonlt], y=[latlt, latrt, latrb, latlb, latlt], pen=True, fill="green", transparency=50)
            # grid = pygmt.xyz2grd(x=mtrue_s['lon'], y=mtrue_s['lat'], z=-mtrue_s[col2plt], region=region, spacing=(0.02, 0.02)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=mtrue_s['lon'], y=mtrue_s['lat'], z=mtrue_s[col2plt], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            if not absflag:
                fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
            else:
                fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=mtrue_s[col2plt].abs(), cmap=True)
            # potency = mom_true_hom['potency'].to_numpy().item()
            # fig.text(text="Potency:", x=region[0]+0.1, y=region[-1]-0.15, font="7p,Helvetica,black", justify="ML")
            # fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.1, y=region[-1]-0.3, font="7p,Helvetica,black", justify="ML")
            
            scale_lon, scale_lat = region[1]-0.4, region[-1]-0.2
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
        
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            
            # fig.plot(x=df['lon'], y=df['lat'], style="s0.15c", fill="cyan", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude"])

        # slip inferred from the inversion under 3D heterogeneity model with syn from same model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+t3D"])
            # maxslip = m_s[col2plt].max()
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_3d['mag'].max()
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s_3d['locking'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # grid = pygmt.xyz2grd(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            if not absflag:
                fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=m_s_3d[col2plt], cmap=True)   # no smoothing or interpolation
            else:
                # fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=m_s_3d[col2plt]-mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
                fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=m_s_3d[col2plt].abs(), cmap=True)   # no smoothing or interpolation
                aa = m_s_3d[col2plt].abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="7p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="7p,Helvetica,black", justify="ML")
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_3d3d.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_3d3d.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_3d3d.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="7p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="7p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="7p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_3d3d.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_3d3d.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_3d3d.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="7p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="7p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="7p,Helvetica,black", justify="ML")
            # grid = pygmt.xyz2grd(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s_3d[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=[0.5], annotation="0.5+f8p", pen="1p,blue")
            # potency = mom_3d3d['potency'].to_numpy().item()
            # fig.text(text="Potency:", x=region[0]+0.1, y=region[-1]-0.15, font="7p,Helvetica,black", justify="ML")
            # fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.1, y=region[-1]-0.3, font="7p,Helvetica,black", justify="ML")

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude"])

    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + slip_str_gt + str(slipmodel) + "_sum_invslip.pdf")


# plot the comparison between true slip and recovery under diff. structure models
plot_resolution_comparison(mtrue_s, m_s_3d3d, assess_col, region, absflag=False, flag_savefig=False)
plot_resolution_comparison(mtrue_s2, m_s_3d3d2, assess_col, region, absflag=False, flag_savefig=False)
plot_resolution_comparison(mtrue_s3, m_s_3d3d3, assess_col, region, absflag=False, flag_savefig=False)

plot_resolution_comparison(mtrue_s, m_s_3d3d-mtrue_s, assess_col, region, absflag=True, flag_savefig=False)
plot_resolution_comparison(mtrue_s2, m_s_3d3d2-mtrue_s2, assess_col, region, absflag=True, flag_savefig=False)
plot_resolution_comparison(mtrue_s3, m_s_3d3d3-mtrue_s3, assess_col, region, absflag=True, flag_savefig=False)


# plot_resolution_comparison(mtrue_s, m_s_swsw, m_s_swsw, m_s_swsw, assess_col, region, flag_savefig=True)
# plot_resolution_comparison(mtrue_s, m_s_homhom, m_s_homhom, m_s_homhom, assess_col, region, flag_savefig=True)


In [ ]:
# # recovered slip along thin profiles
# mtrue_s['xf'], mtrue_s['yf'] = rot_xy(mtrue_s['xf'], mtrue_s['yf'], -rot) 

# # Option 1: Thin slice (10 km thickness)
# bin_width_km = 10
# thickness_km = 10

# prof1_3d = utp.extract_directional_profile(x_coords=mtrue_s['xf'], y_coords=mtrue_s['yf'], 
#                                         values=m_s_3d3d[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=-40, profile_center_y=40,
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof2_3d = utp.extract_directional_profile(x_coords=mtrue_s['xf'], y_coords=mtrue_s['yf'], 
#                                         values=m_s_3d3d[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=0, profile_center_y=0, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof3_3d = utp.extract_directional_profile(x_coords=mtrue_s['xf'], y_coords=mtrue_s['yf'], 
#                                         values=m_s_3d3d[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=40, profile_center_y=-20, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)

# prof1_3d2 = utp.extract_directional_profile(x_coords=loc_f2['xf'], y_coords=loc_f2['yf'], 
#                                         values=m_s_3d3d2[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=-40, profile_center_y=40,
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof2_3d2 = utp.extract_directional_profile((x_coords=loc_f2['xf'], y_coords=loc_f2['yf'], 
#                                         values=m_s_3d3d2[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=0, profile_center_y=0, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof3_3d2 = utp.extract_directional_profile((x_coords=loc_f2['xf'], y_coords=loc_f2['yf'], 
#                                         values=m_s_3d3d2[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=40, profile_center_y=-20, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)

# prof1_3d3 = utp.extract_directional_profile(x_coords=loc_f3['xf'], y_coords=loc_f3['yf'], 
#                                         values=m_s_3d3d3[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=-40, profile_center_y=40,
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof2_3d3 = utp.extract_directional_profile(x_coords=loc_f3['xf'], y_coords=loc_f3['yf'], 
#                                         values=m_s_3d3d3[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=0, profile_center_y=0, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)
# prof3_3d3 = utp.extract_directional_profile(x_coords=loc_f3['xf'], y_coords=loc_f3['yf'], 
#                                         values=m_s_3d3d3[assess_col].values, azimuth_deg=45,
#                                         profile_center_x=40, profile_center_y=-20, 
#                                         profile_length=250, thickness_km=thickness_km, bin_width_km=bin_width_km)

# profile_sets = [
#     [prof1_3d, prof1_3d2, prof1_3d3],  # Location A
#     [prof2_3d, prof2_3d2, prof2_3d3],  # Location B  
#     [prof3_3d, prof3_3d2, prof3_3d3]   # Location C
# ]
# utp.plot_profile_enhanced(profile_sets, mtrue_s['xf'].values, mtrue_s['yf'].values,
#                       m_s_homhom[assess_col].values, m_s_swsw[assess_col].values,
#                       m_s_3d3d[assess_col].values, mtrue_s[assess_col].values)

In [ ]:
def plot_homhetvstrue_slip(m_s_hom, m_s, mtrue_s, col2plt, region, struc_str_for, isreftrue=True, flag_savefig=False):

    slipsybsz = "c0.09c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the hom-true slip vs. het-true slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # True Slip
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tRef slip"])
            # fig.coast(region=region, projection="M?", frame=["af", "+tRef slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            maxslip = 1
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
            # grid = pygmt.xyz2grd(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation

            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude"])
        
        # ABS difference between the homogeneous and true model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHom - Ref"])
            maxdiff = np.max([(m_s_hom[col2plt] - mtrue_s[col2plt]).abs().max(), (m_s[col2plt] - mtrue_s[col2plt]).abs().max()])
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = 0.5
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=m_s_hom[col2plt] - mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=(m_s_hom[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)            
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            #if the reference slip is true slip, show the rmse
            if isreftrue:
                aa = (m_s_hom[col2plt] - mtrue_s[col2plt]).abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="7p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="7p,Helvetica,black", justify="ML")
                metrics_hom = utp.SlipRecoveryMetrics(mtrue_s, m_s_hom, mtrue_s)
                # if slipmodel == 2:   
                #     rmse_slip1 = metrics_hom.rmse(mask_shallow, col=assess_col)
                #     rmse_slip2 = metrics_hom.rmse(mask_deep, col=assess_col)
                #     rmse_gap = metrics_hom.rmse(mask_gap, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="7p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="7p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="7p,Helvetica,black", justify="ML")
                # if slipmodel == 3:
                #     rmse_gap1 = metrics_hom.rmse(mask_gap1, col=assess_col)
                #     rmse_gap2 = metrics_hom.rmse(mask_gap2, col=assess_col)
                #     rmse_slip = metrics_hom.rmse(mask_inter, col=assess_col)
                #     fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="7p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="7p,Helvetica,black", justify="ML")
                #     fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="7p,Helvetica,black", justify="ML")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lDifference"])

        # ABS difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHet - Ref"])
            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap=colormap, series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 1/10], background=True, reverse=True)
            # pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/10], background=True, reverse=True)        
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=mtrue_s['lon'], y=mtrue_s['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=m_s[col2plt] - mtrue_s[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=mtrue_s['lon'], y=mtrue_s['lat'], style=slipsybsz, fill=(m_s[col2plt] - mtrue_s[col2plt]).abs(), cmap=True)   # no smoothing or interpolation

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")
            
            #if the reference slip is true slip, show the rmse
            if isreftrue:
                aa = (m_s[col2plt] - mtrue_s[col2plt]).abs().to_numpy()
                rmse_global = np.sqrt(np.mean(aa ** 2))
                fig.text(text="RMSE", x=region[0]+0.1, y=region[-2]+0.75, font="7p,Helvetica,black", justify="ML")
                fig.text(text=f"Global: {rmse_global:.3f}", x=region[0]+0.1, y=region[-2]+0.6, font="7p,Helvetica,black", justify="ML")
                metrics_het = utp.SlipRecoveryMetrics(mtrue_s, m_s, mtrue_s)
                if slipmodel == 2:   
                    rmse_slip1 = metrics_het.rmse(mask_shallow, col=assess_col)
                    rmse_slip2 = metrics_het.rmse(mask_deep, col=assess_col)
                    rmse_gap = metrics_het.rmse(mask_gap, col=assess_col)
                    fig.text(text=f"Shallow: {rmse_slip1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="7p,Helvetica,black", justify="ML")
                    fig.text(text=f"Deep: {rmse_slip2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="7p,Helvetica,black", justify="ML")
                    fig.text(text=f"Intermediate: {rmse_gap:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="7p,Helvetica,black", justify="ML")
                if slipmodel == 3:
                    rmse_gap1 = metrics_het.rmse(mask_gap1, col=assess_col)
                    rmse_gap2 = metrics_het.rmse(mask_gap2, col=assess_col)
                    rmse_slip = metrics_het.rmse(mask_inter, col=assess_col)
                    fig.text(text=f"Shallow: {rmse_gap1:.3f}", x=region[0]+0.1, y=region[-2]+0.45, font="7p,Helvetica,black", justify="ML")
                    fig.text(text=f"Deep: {rmse_gap2:.3f}", x=region[0]+0.1, y=region[-2]+0.3, font="7p,Helvetica,black", justify="ML")
                    fig.text(text=f"Intermediate: {rmse_slip:.3f}", x=region[0]+0.1, y=region[-2]+0.15, font="7p,Helvetica,black", justify="ML")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["af", "x+lDifference"])
                
    fig.show()

    if flag_savefig:
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_slipdiff.pdf")

## difference relative to the true model
# plot_homhetvstrue_slip(m_s_homhom, m_s_homhom, mtrue_s, assess_col, region, homo_str, flag_savefig=False)    
# plot_homhetvstrue_slip(m_s_swsw, m_s_3d3d, mtrue_s, assess_col, region, homo_str, flag_savefig=False)    

## difference relative to the homo model
# plot_homhetvstrue_slip(m_s_swsw, m_s_3d3d, m_s_homhom, assess_col, region, homo_str, isreftrue=False, flag_savefig=False)  

# Under the same observation (displacements) from a heterogeneous structure, how does inversion vary if ignoring the structure?

* In other words, the inversion is based on the same synthetic data, which is generated from the respective heterogeneous structure (either Slab/Wedge model or amplified 3D model)


In [ ]:
def plot_homvshet_slip(m_s_hom, m_s, col2plt, region, struc_str_for, mom_hom, mom, flag_savefig=False):

    slipsybsz = "c0.09c"
    # colormap = "lajolla"
    colormap = "viridis"

    # plot the hom slip vs. het slip on the fault, GMT style
    fig = pygmt.Figure()
    with fig.subplot(nrows=1, ncols=3, figsize=("15c", "10c"), autolabel=False, frame=["a1f0.2", "WSne"],
                    margins=["0.1c", "0.2c"], sharey=True):

        pygmt.config(MAP_FRAME_TYPE="plain", FONT_TITLE="10p,Helvetica-Bold,black", MAP_TITLE_OFFSET="-0.2c", MAP_SCALE_HEIGHT="3p",
                    FONT_ANNOT_PRIMARY="6p", FONT_ANNOT_SECONDARY="10p")

        # Slip inferred from the homogeneous model
        with fig.set_panel(panel=[0, 0]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHom. inv."])
            # fig.coast(region=region, projection="M?", frame=["af", "+tTrue slip"], borders=1, 
            #           shorelines="0.25p,black", area_thresh=4000) #frame="af", 
            maxslip = 1
            # maxslip = mtrue_s[col2plt].max()
            # maxslip = m_s_hom[col2plt].max()
            # maxslip = max(m_s_hom[col2plt].max(), m_s[col2plt].max(), mtrue_s[col2plt].max())
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=m_s['lon'], y=m_s['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.04, 0.04)) # no smoothing
            # grid = pygmt.xyz2grd(x=m_s['lon'], y=m_s['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=m_s['lon'], y=m_s['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=m_s['lon'], y=m_s['lat'], style=slipsybsz, fill=m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=m_s['lon'], y=m_s['lat'], style=slipsybsz, fill=m_s_hom[col2plt].abs(), cmap=True)   # no smoothing or interpolation

            scale_lon, scale_lat = region[1]-0.5, region[-1]-0.25
            mapscale_str = "g" + str(scale_lon) + "/" + str(scale_lat) + "c" + str(scale_lat) + "+w50k"
            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            potency = mom_hom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.1, y=region[-1]-0.15, font="7p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.1, y=region[-1]-0.3, font="7p,Helvetica,black", justify="ML")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude"]) #, "y+l(mm)"

        # slip inferred from the heterogeneous model
        with fig.set_panel(panel=[0, 1]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHet. inv."])
            pygmt.makecpt(cmap=colormap, series=[0, maxslip, maxslip/20], background=True, reverse=True)    #m_s_hom[col2plt].max()
            # grid = pygmt.xyz2grd(x=m_s['lon'], y=m_s['lat'], z=m_s['locking'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # grid = pygmt.xyz2grd(x=m_s['lon'], y=m_s['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=m_s['lon'], y=m_s['lat'], z=m_s_hom[col2plt], region=region, spacing=(0.05, 0.05))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.05, 0.05))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True)  # smoothing and interpolation

            fig.plot(x=m_s['lon'], y=m_s['lat'], style=slipsybsz, fill=m_s[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=m_s['lon'], y=m_s['lat'], style=slipsybsz, fill=m_s[col2plt].abs(), cmap=True)   # no smoothing or interpolation

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            potency = mom['potency'].to_numpy().item()
            fig.text(text="Potency:", x=region[0]+0.1, y=region[-1]-0.15, font="7p,Helvetica,black", justify="ML")
            fig.text(text=f"{potency:.3e} m@+3@+", x=region[0]+0.1, y=region[-1]-0.3, font="7p,Helvetica,black", justify="ML")
            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                fig.colorbar(frame=["af", "x+lMagnitude"])

        # Difference between the heterogeneous and homogeneous model
        with fig.set_panel(panel=[0, 2]):
            fig.basemap(region=region, projection="M?", frame=["a1f0.2", "+tHet - Hom"])
            if struc_str_for == homo_str:
                maxdiff = 10
            else:
                maxdiff = (m_s[col2plt] - m_s_hom[col2plt]).abs().max()
                # maxdiff = 0.5
                print(maxdiff)
            maxdiff = 0.3

            # pygmt.makecpt(cmap="polar", series=[-maxdiff, maxdiff, maxdiff/5], background=True, reverse=True)
            pygmt.makecpt(cmap="roma", series=[-maxdiff, maxdiff, maxdiff/10], background=True, reverse=False)
            # pygmt.makecpt(cmap="lajolla", series=[0, maxdiff, maxdiff/10], background=True, reverse=True)
            # fig.basemap(region=region, projection="M?", frame=["af", "+tlog_{10}{Het / Hom. slip}"])
            # maxdiff = np.ceil(m_s_mag['diff_ratio'].abs().max())
            # pygmt.makecpt(cmap="polar", series=[-0.5, 0.5, 0.05], background=True, reverse=True)
            # grid = pygmt.xyz2grd(x=m_s['lon'], y=m_s['lat'], z=m_s['s_dip']-mtrue_s['s_dip'], region=region, spacing=(0.05, 0.05)) # no smoothing
            # By averaging values in blocks, blockmean can help smooth out noise or reduce small-scale fluctuations in the data.
            # data_bmedian = pygmt.blockmedian(x=m_s['lon'], y=m_s['lat'], z=m_s[col2plt]-m_s_hom[col2plt], region=region, spacing=(0.025, 0.025))
            # data_bmedian = pygmt.blockmedian(x=m_s['lon'], y=m_s['lat'], z=m_s_mag['diff'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # fig.grdimage(grid=grid, cmap=True, nan_transparent=True, interpolation=None)

            # fig.plot(x=m_s['lon'], y=m_s['lat'], style=slipsybsz, fill=(m_s[col2plt] - m_s_hom[col2plt]).abs(), cmap=True)   # no smoothing or interpolation
            fig.plot(x=m_s['lon'], y=m_s['lat'], style=slipsybsz, fill=m_s[col2plt] - m_s_hom[col2plt], cmap=True)   # no smoothing or interpolation
            # fig.plot(x=m_s['lon'], y=m_s['lat'], style="c0.125c", fill=m_s_mag['diff_ratio'], cmap=True)   # no smoothing or interpolation

            fig.coast(borders=None, shorelines="0.25p,black", area_thresh=20, map_scale=mapscale_str)
            # data_bmedian = pygmt.blockmedian(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.025, 0.025))
            # grid = pygmt.surface(data=data_bmedian, region=region, spacing=(0.025, 0.025))
            # grid = pygmt.xyz2grd(x=df_plate['lon'], y=df_plate['lat'], z=df_plate['z'], region=region, spacing=(0.04, 0.04)) # no smoothing
            # fig.grdcontour(grid=grid, levels=5, limit="-100/-5", annotation="20+f8p", pen="0.5p,darkgray") 
            # fig.plot(x=trench['lon'], y=trench['lat'], pen="1p,dimgray", style="f0.4i/0.075i+l+t", fill="dimgray")
            # fig.plot(x=volcano['lon'], y=volcano['lat'], style="t0.15c", fill="black", pen="0.25p,black")

            with pygmt.config(FONT_ANNOT_PRIMARY="8p"):
                # fig.colorbar(frame=["af", "x+lAbs. difference"])
                fig.colorbar(frame=["a0.1f0.02", "x+lDifference"])

    fig.show()

    if flag_savefig:
        # print('yes')
        # if struc_str_for == homo_str:
        #     fig.savefig(figpath + meshname + "_hom_slip.pdf")
        # else:
        fig.savefig(figpath + meshname + struc_str_for + str(slipmodel) + "_hethomslip.pdf")


## difference relative to the true model
# plot_homhetvstrue_slip(m_s_swhom, m_s_swsw, mtrue_s, assess_col, region, homo_str, flag_savefig=False)   

## slip difference under inversion using hom or het models based on same data from het models
# plot_homvshet_slip(m_s_homhom, m_s_homhom, assess_col, region, homo_str, flag_savefig=False)    
# plot_homvshet_slip(m_s_swhom, m_s_swsw, assess_col, region, sw_str, mom_swhom, mom_swsw, flag_savefig=True)   

## same thing, but for 3D model
# plot_homhetvstrue_slip(m_s_3dhom, m_s_3d3d, mtrue_s, assess_col, region, homo_str, flag_savefig=False)  
plot_homvshet_slip(m_s_3dhom, m_s_3d3d, assess_col, region, het3d_str, mom_3dhom, mom_3d3d, flag_savefig=False) 

plot_homvshet_slip(m_s_3dhom2, m_s_3d3d2, assess_col, region, het3d_str, mom_3dhom2, mom_3d3d2, flag_savefig=False) 

plot_homvshet_slip(m_s_3dhom3, m_s_3d3d3, assess_col, region, het3d_str, mom_3dhom3, mom_3d3d3, flag_savefig=False) 
